## Fine-tuning эмбеддингов

In [37]:
%load_ext autoreload
%autoreload 3

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [38]:
from torch.optim import Adam
from torch.utils.data import DataLoader



from sneakers_hse.data.utils.s3_tools import S3Client

import polars as pl
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path
import os

In [39]:
PROJECT_ROOT_PATH = Path(os.getenv('PROJECT_ROOT_PATH'))
# # Скачиваю эмбеддинги из s3
# s3 = S3Client(aws_access_key_id=os.getenv("AWS_ACCESS_KEY"),
#               aws_secret_access_key=os.getenv("AWS_SECRET_KEY"))
# s3._download_one(
#     bucket_name='sneakers-hse-images-test',
#     s3_key='dinov2/embeddings.parquet.gzip', # Путь пишем с учётом модели, т.е. каждой модели будет соответстовать отдельный инстанс ChromaDB
#     local_path=str(PROJECT_ROOT_PATH / 'data/04_dinov2_embeddings.parquet.gzip')
# )

In [40]:
df = pl.read_parquet(PROJECT_ROOT_PATH / "data/04_dinov2_embeddings.parquet.gzip")

embeddings = np.stack(df["embedding"].to_list()).astype("float32")
labels = np.array(df["class"].to_list())
paths = df["path"].to_list()
df['class'].value_counts()
train_df_pre = pl.read_csv(PROJECT_ROOT_PATH / 'data/03_yolo_preprocessed/train_images.csv')
test_df = pl.read_csv(PROJECT_ROOT_PATH / 'data/03_yolo_preprocessed/test_images.csv')

train_df, val_df = train_test_split(
    train_df_pre,
    test_size=0.2,
    stratify=train_df_pre["sneaker_class"],
    random_state=42
)

display(train_df.head(), train_df.shape)
display(val_df.head(), val_df.shape)
display(test_df.head(), test_df.shape)
sql = pl.SQLContext()
sql.register_globals()
df = sql.execute(
    '''
select
    df.*
    , case when train_df.path is not null then 'train'
            when val_df.path is not null then 'val'
            when test_df.path is not null then 'test'
            end as sample_part
from df
left join train_df using(path)
left join val_df using(path)
left join test_df using(path)
'''
).collect()
df

,path,sneaker_class,corrupted_flg
i64,str,str,i64
2288,"""Nike Кеды Dunk Low Retro/cloth…","""Nike Кеды Dunk Low Retro""",0
10855,"""PUMA Кроссовки Puma Morphic/or…","""PUMA Кроссовки Puma Morphic""",0
7477,"""Kari Кроссовки/clothing_0_190.…","""Kari Кроссовки""",0
4230,"""Reebok Кроссовки CLASSIC LEATH…","""Reebok Кроссовки CLASSIC LEATH…",0
2188,"""Nike Кеды Dunk Low Retro/cloth…","""Nike Кеды Dunk Low Retro""",0


(8772, 4)

,path,sneaker_class,corrupted_flg
i64,str,str,i64
7639,"""Vans Кеды Knu Skool/clothing_0…","""Vans Кеды Knu Skool""",0
2591,"""Reebok Кроссовки CLASSIC NYLON…","""Reebok Кроссовки CLASSIC NYLON""",0
5027,"""Nike Кроссовки AIR MAX 90/clot…","""Nike Кроссовки AIR MAX 90""",0
2762,"""Under Armour Кроссовки UA CHAR…","""Under Armour Кроссовки UA CHAR…",0
240,"""Vans Кеды Upland/clothing_0_62…","""Vans Кеды Upland""",0


(2193, 4)

,path,sneaker_class,corrupted_flg
i64,str,str,i64
14,"""Vans Кеды Upland/clothing_0_16…","""Vans Кеды Upland""",0
32,"""Vans Кеды Upland/clothing_0_21…","""Vans Кеды Upland""",0
44,"""Vans Кеды Upland/orig_216_real…","""Vans Кеды Upland""",0
80,"""Vans Кеды Upland/shoe_3_100_re…","""Vans Кеды Upland""",0
87,"""Vans Кеды Upland/clothing_0_27…","""Vans Кеды Upland""",0


(300, 4)

path,class,embedding,sample_part
str,str,list[f64],str
"""Vans Кеды Upland/clothing_0_26…","""Vans Кеды Upland""","[-1.930393, -0.157552, … 0.300103]","""train"""
"""Vans Кеды Upland/clothing_0_57…","""Vans Кеды Upland""","[-1.749115, -1.480964, … -0.966375]","""train"""
"""Vans Кеды Upland/orig_45.jpeg""","""Vans Кеды Upland""","[0.503809, -0.948843, … -3.250608]","""train"""
"""Vans Кеды Upland/clothing_0_0.…","""Vans Кеды Upland""","[-1.441995, -2.107342, … -0.808429]","""train"""
"""Vans Кеды Upland/clothing_0_23…","""Vans Кеды Upland""","[-0.845633, -2.200308, … -1.242736]","""train"""
…,…,…,…
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[0.131167, -0.318168, … -0.66955]","""train"""
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[1.267569, -3.275375, … -2.853185]","""train"""
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[1.495174, -1.613851, … -1.374945]","""train"""


In [41]:
train_df = df.filter(pl.col('sample_part') == 'train')
val_df = df.filter(pl.col('sample_part') == 'val')
test_df = df.filter(pl.col('sample_part') == 'test')
train_df

path,class,embedding,sample_part
str,str,list[f64],str
"""Vans Кеды Upland/clothing_0_26…","""Vans Кеды Upland""","[-1.930393, -0.157552, … 0.300103]","""train"""
"""Vans Кеды Upland/clothing_0_57…","""Vans Кеды Upland""","[-1.749115, -1.480964, … -0.966375]","""train"""
"""Vans Кеды Upland/orig_45.jpeg""","""Vans Кеды Upland""","[0.503809, -0.948843, … -3.250608]","""train"""
"""Vans Кеды Upland/clothing_0_0.…","""Vans Кеды Upland""","[-1.441995, -2.107342, … -0.808429]","""train"""
"""Vans Кеды Upland/clothing_0_23…","""Vans Кеды Upland""","[-0.845633, -2.200308, … -1.242736]","""train"""
…,…,…,…
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[-1.853474, -1.394531, … -1.656443]","""train"""
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[0.131167, -0.318168, … -0.66955]","""train"""
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[1.267569, -3.275375, … -2.853185]","""train"""


## Обучаю модель

In [69]:
from sneakers_hse.model.triplet_loss import EmbeddingDataset
from pytorch_metric_learning.samplers import MPerClassSampler

train_dataset = EmbeddingDataset(train_df)
sampler = MPerClassSampler(labels=train_dataset.labels, m=4, batch_size=64, length_before_new_iter=1_000)
train_dataloader = DataLoader(train_dataset, sampler=sampler)

val_dataset = EmbeddingDataset(val_df)
val_dataloader = DataLoader(val_dataset, batch_size=64)

In [70]:
for x, y in train_dataloader:
    print(x.shape)
    break

torch.Size([1, 768])


In [71]:
from pytorch_metric_learning import losses, miners

loss_fn = losses.TripletMarginLoss(margin=0.2)

miner = miners.TripletMarginMiner(
    margin=0.2,
    type_of_triplets="semihard"
)

In [72]:
import torch.nn as nn

input_dim = 768
embedding_dim = 768
model = nn.Sequential(
    nn.Linear(input_dim, 1024),
    nn.ReLU(),
    nn.Linear(1024, 1024),
    nn.ReLU(),
    nn.Linear(1024, embedding_dim)
)
model = model.train()
model

Sequential(
  (0): Linear(in_features=768, out_features=1024, bias=True)
  (1): ReLU()
  (2): Linear(in_features=1024, out_features=1024, bias=True)
  (3): ReLU()
  (4): Linear(in_features=1024, out_features=768, bias=True)
)

In [73]:
import torch

val_df


path,class,embedding,sample_part,new_embedding
str,str,list[f64],str,"array[f32, 768]"
"""Vans Кеды Upland/orig_28.jpeg""","""Vans Кеды Upland""","[-1.147287, -1.009799, … 0.806448]","""val""","[-0.146398, -0.042614, … -0.243736]"
"""Vans Кеды Upland/clothing_0_94…","""Vans Кеды Upland""","[-1.412233, -1.431905, … -0.192771]","""val""","[-0.282348, 0.055301, … -0.280446]"
"""Vans Кеды Upland/orig_115.jpeg""","""Vans Кеды Upland""","[-0.926339, 0.773644, … -0.431636]","""val""","[-0.144878, -0.039798, … -0.335569]"
"""Vans Кеды Upland/clothing_0_19…","""Vans Кеды Upland""","[-1.072493, -4.571333, … -0.351652]","""val""","[-0.273616, 0.018096, … -0.343255]"
"""Vans Кеды Upland/clothing_0_61…","""Vans Кеды Upland""","[-1.447895, -0.958692, … -0.995994]","""val""","[-0.248794, 0.080571, … -0.309518]"
…,…,…,…,…
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[1.329301, -1.360057, … -1.224334]","""val""","[-0.156632, -0.019594, … -0.180783]"
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[-0.504423, -1.179003, … -0.584849]","""val""","[-0.133833, 0.014703, … -0.202139]"
"""Under Armour Кроссовки UA Phan…","""Under Armour Кроссовки UA Phan…","[-0.133964, 0.777298, … -0.496436]","""val""","[0.071868, -0.067333, … -0.311711]"


In [86]:
from tqdm import tqdm
import chromadb
from sneakers_hse.metrics import get_neighbors
from torch.optim import Adam

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
optimizer = Adam(model.parameters(), lr=1e-3)

num_epochs = 10


client = chromadb.Client()
for epoch in range(num_epochs):

    # Train
    model.train()
    total_loss = 0
    for embeddings, labels in tqdm(train_dataloader):
        
        embeddings = embeddings.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        projected = model(embeddings)

        hard_pairs = miner(projected, labels)
        loss = loss_fn(projected, labels, hard_pairs)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch}: loss={total_loss:.4f}")

    # Val
    model.eval()
    collection = client.get_or_create_collection(
            "new_embeddings",
            metadata={"hnsw:space": "cosine"}
        )
    new_embeddings = []
    for embedding, labels in val_dataloader:
        with torch.no_grad():
            new_embedding = model(embedding)
        new_embeddings.append(new_embedding)
    new_embeddings = torch.concat(new_embeddings).numpy()

    val_df = val_df.with_columns(new_embedding = new_embeddings)
    collection.add(val_df['path'].to_list(),
                   embeddings=new_embeddings,
                   metadatas=[{"class": c} for c in val_df['class'].to_list()])
    neighbors = get_neighbors(collection, new_embeddings)
    val_df = val_df.with_columns(neighbors=neighbors)
    val_df = val_df.with_columns(
        pl.struct(["neighbors", "class"]).map_elements(
            lambda row: [
                1 if n == row["class"] else 0
                for n in row["neighbors"]
            ]
        ).alias("hit_flg")
    )
    for k in [1, 5, 10]:
        print(f"\nMetrics @ {k}")
        df_k = df.with_columns(hits_k = pl.col("hit_flg").list.slice(0, k))\
            .with_columns(
                pos = pl.int_ranges(1, pl.col("hits_k").list.len() + 1),
                hit_at_k = pl.col("hits_k").list.max(),
                precision_at_k = pl.col("hits_k").list.mean()
            )
        display(df_k.select(
            pl.col('hit_at_k').mean(),
            pl.col('precision_at_k').mean()
        ))
    client.delete_collection('new_embeddings')

    

100%|██████████| 960/960 [00:03<00:00, 263.37it/s]


Epoch 0: loss=0.0000
{'ids': [['Vans Кеды Upland/orig_28.jpeg', 'Vans Кеды Upland/orig_276.jpeg', 'Vans Кеды Upland/orig_133.jpeg', 'Reebok Кроссовки CLASSIC NYLON/orig_76.jpeg', 'Vans Кеды Upland/orig_36.jpeg', 'Vans Кеды Upland/orig_214.jpeg', 'PUMA Кроссовки Hypnotic LS/orig_42.jpeg', 'Vans Кеды Upland/clothing_0_163.jpeg', 'adidas Кроссовки DROPSET 3 TRAINER/orig_94.jpeg', 'X-Plode Кеды/orig_5.jpeg', 'adidas Кроссовки RUNFALCON 5/orig_109.jpeg'], ['Vans Кеды Upland/clothing_0_94.jpeg', 'Reebok Кроссовки CLASSIC LEATHER/orig_192.jpeg', 'Vans Кеды Upland/clothing_0_125.jpeg', 'Vans Кеды Upland/clothing_0_167.jpeg', 'PUMA Кроссовки Hypnotic LS/clothing_0_124.jpeg', 'Vans Кеды Upland/clothing_0_254.jpeg', 'Under Armour Кроссовки UA CHARGED EDGE/clothing_0_75.jpeg', 'Under Armour Кроссовки UA CHARGED EDGE/clothing_0_113.jpeg', 'Vans Кеды Upland/orig_27.jpeg', 'Vans Кеды Upland/clothing_0_148.jpeg', 'Nike Кеды Dunk Low Retro/orig_283.jpeg'], ['Vans Кеды Upland/orig_115.jpeg', 'Vans Кеды 

TypeError: 'NoneType' object is not subscriptable

In [76]:
collection.get('Vans Кеды Upland/orig_28.jpeg', include=['embeddings'])

{'ids': ['Vans Кеды Upland/orig_28.jpeg'],
 'embeddings': array([[-1.47078365e-01,  2.57406294e-01,  2.37518698e-01,
          1.37765789e-02, -1.15086406e-01, -3.29904079e-01,
         -1.36756375e-01, -3.09500452e-02,  2.58732378e-01,
          3.86957228e-02,  1.57681659e-01,  5.14240026e-01,
          1.40181437e-01, -2.82009572e-01,  1.39537454e-01,
         -2.67069399e-01, -2.57970840e-01,  3.41405123e-01,
          9.87437293e-02, -1.09773658e-01,  1.60057038e-01,
         -1.82513207e-01,  2.22383648e-01,  2.28469595e-01,
          2.11753979e-01, -1.14118271e-01,  4.72986256e-04,
         -5.81337139e-02, -1.68772608e-01,  1.84595495e-01,
          5.28222285e-02, -3.48801166e-01,  2.42122501e-01,
         -9.92079526e-02,  1.55692726e-01, -4.76329923e-02,
         -6.33245334e-02, -1.10706411e-01,  2.04372749e-01,
          3.26428446e-03, -1.96937442e-01,  5.05853295e-02,
         -6.44004270e-02, -1.75388217e-01,  8.60712752e-02,
          3.29416478e-03, -2.13989303e-01, 